# Custom pipeline: Faster R-CNN with tiled training and tiled prediction

In [ ]:
!pip install -q pycocotools

In [ ]:
import shutil, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import yaml
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision.ops import batched_nms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO
from google.colab import auth, drive

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU - turn on the GPU runtime')

auth.authenticate_user()
email = subprocess.getoutput('gcloud config get-value account')
print(f'Current Google User: {email}')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/datasets/hw2_train-s3_val-s3/'

In [ ]:
OUT = Path('/content/drive/MyDrive/hw2_runs/frcnn_tiled')  # checkpoints, metrics, submission
RUN_NAME = 'tiled'                                          # checkpoint: OUT / f'{RUN_NAME}_{EPOCHS}ep.pt'

# Start from the full-image Faster R-CNN weights (much faster than from COCO). Set to None to start from COCO weights.
INIT_CKPT = Path('/content/drive/MyDrive/final_6ep.pt')

EPOCHS = 4
LR = 0.005 if INIT_CKPT is None else 0.0025  # lower when the weights are already trained on this data
BATCH_SIZE = 2           # lower MIN_SIZE to 800 if you get CUDA out of memory
MIN_SIZE, MAX_SIZE = 1024, 1824
MAX_DET = 300            # scenes are dense
SCORE_THRESH = 0.01      # low threshold is standard for mAP
RUN_PARAMS = {'box_positive_fraction': 0.5}  # roi_pos_50, the winner of the 02 study

# Training crops
CROP_MIN, CROP_MAX = 640, 800  # crop side, random per sample
FULL_FRAME_P = 0.2             # share of samples that stay full frames
OBJECT_CENTRED_P = 0.8         # share of crops placed around a random object (the rest are uniform)
MIN_VISIBLE = 0.5              # a box cut by the crop is kept if at least this share of it is inside

# Tiled prediction
TILE = 640               # tile side in original pixels; enlarged to MIN_SIZE by the model
TILE_OVERLAP = 0.2
TILE_BATCH = 4           # tiles per forward pass
EDGE_MARGIN = 2          # px: tile boxes this close to an inner tile edge are treated as cut and dropped
MERGE_IOU = 0.6          # class-wise NMS IoU for merging tiles and the full pass

OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
drive.mount('/content/drive', force_remount=True)
if INIT_CKPT is not None:
    assert INIT_CKPT.exists(), f'{INIT_CKPT} not found: fix the path or set INIT_CKPT = None'

In [ ]:
# Copy the dataset from Drive to the local disk, so training doesn't read every image over the network
ROOT = Path('/content/hw2_data')

def copy_one(src):
    dst = ROOT / src.relative_to(DRIVE_ROOT)
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_name(dst.name + '.part')
    shutil.copy(src, temp)
    temp.replace(dst)

files = [p for p in Path(DRIVE_ROOT).rglob('*') if p.is_file()]
print(f'Copying {len(files):,} files...')
with ThreadPoolExecutor(16) as pool:
    list(tqdm(pool.map(copy_one, files), total=len(files)))

## Data

In [ ]:
names = yaml.safe_load(open(ROOT / 'train' / 'data.yaml'))['names']
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names
NUM_CLASSES = len(names)


class CocoFrames(Dataset):
    """Full frames with boxes from annotations_coco.json (same as the 02 notebook)."""
    def __init__(self, split, limit=None):
        self.coco = COCO(str(ROOT / split / 'annotations_coco.json'))

        # Remove annotations with an unlisted category or zero size, for training and evaluation alike
        anns = self.coco.dataset['annotations']
        good = [a for a in anns if a['category_id'] in self.coco.cats and a['bbox'][2] > 0 and a['bbox'][3] > 0]
        for a in good:
            a.setdefault('iscrowd', 0)
            a.setdefault('area', a['bbox'][2] * a['bbox'][3])
        self.coco.dataset['annotations'] = good
        self.coco.createIndex()

        # Images are in <split>/ or <split>/images/. Keep only frames that exist on disk.
        for self.folder in [ROOT / split, ROOT / split / 'images']:
            self.ids = [i for i, im in sorted(self.coco.imgs.items()) if (self.folder / im['file_name']).exists()]
            if self.ids:
                break
        assert self.ids, f'No images from the {split} json were found on disk'
        total = len(self.ids)
        if limit and limit < total:  # evenly spaced, so every clip stays represented
            self.ids = [self.ids[k] for k in np.linspace(0, total - 1, limit).astype(int)]

        # COCO category id -> 0-based class index from data.yaml
        cats = sorted(self.coco.cats.values(), key=lambda c: c['id'])
        if all(c['name'] in names for c in cats):
            self.cat2cls = {c['id']: names.index(c['name']) for c in cats}
        else:
            print('WARNING: json category names differ from data.yaml, matching them by order')
            self.cat2cls = {c['id']: k for k, c in enumerate(cats)}
        print(f'{split}: using {len(self.ids):,} frames ({len(self.coco.imgs):,} in json, {total:,} on disk), '
              f'dropped {len(anns) - len(good):,} bad annotations')

    def __len__(self):
        return len(self.ids)

    def path(self, image_id):
        return self.folder / self.coco.imgs[image_id]['file_name']

    def boxes_and_labels(self, image_id):
        boxes, labels = [], []
        for a in self.coco.imgToAnns[image_id]:
            x, y, w, h = a['bbox']
            if not a['iscrowd']:
                boxes.append([x, y, x + w, y + h])
                labels.append(self.cat2cls[a['category_id']] + 1)  # 0 is background
        return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), torch.tensor(labels, dtype=torch.int64)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        image = TF.to_tensor(Image.open(self.path(image_id)).convert('RGB'))
        boxes, labels = self.boxes_and_labels(image_id)
        return image, {'boxes': boxes, 'labels': labels, 'image_id': image_id}


class CropFrames(Dataset):
    """Training samples: a random CROP_MIN-CROP_MAX px crop of each frame (or the full frame), with a horizontal flip."""
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def crop_window(self, W, H, boxes):
        side = int(torch.randint(CROP_MIN, CROP_MAX + 1, ()))
        cw, ch = min(side, W), min(side, H)
        if len(boxes) and torch.rand(()) < OBJECT_CENTRED_P:
            # put a random object somewhere inside the crop, not always in the centre
            b = boxes[torch.randint(len(boxes), ())]
            cx, cy = float(b[0] + b[2]) / 2, float(b[1] + b[3]) / 2
            x0 = cx - float(torch.empty(()).uniform_(0.1, 0.9)) * cw
            y0 = cy - float(torch.empty(()).uniform_(0.1, 0.9)) * ch
        else:
            x0 = float(torch.rand(())) * (W - cw)
            y0 = float(torch.rand(())) * (H - ch)
        x0 = int(min(max(x0, 0), W - cw))
        y0 = int(min(max(y0, 0), H - ch))
        return x0, y0, x0 + cw, y0 + ch

    def __getitem__(self, idx):
        image_id = self.base.ids[idx]
        image = Image.open(self.base.path(image_id)).convert('RGB')
        boxes, labels = self.base.boxes_and_labels(image_id)

        if torch.rand(()) >= FULL_FRAME_P:
            x0, y0, x1, y1 = self.crop_window(*image.size, boxes)
            image = image.crop((x0, y0, x1, y1))
            clipped = boxes.clone()
            clipped[:, [0, 2]] = clipped[:, [0, 2]].clamp(x0, x1) - x0
            clipped[:, [1, 3]] = clipped[:, [1, 3]].clamp(y0, y1) - y0
            area = lambda b: (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
            w, h = clipped[:, 2] - clipped[:, 0], clipped[:, 3] - clipped[:, 1]
            keep = (area(clipped) >= MIN_VISIBLE * area(boxes)) & (w >= 2) & (h >= 2)
            boxes, labels = clipped[keep], labels[keep]

        image = TF.to_tensor(image)
        if torch.rand(()) < 0.5:  # horizontal flip
            image = image.flip(-1)
            boxes[:, [0, 2]] = image.shape[-1] - boxes[:, [2, 0]]
        return image, {'boxes': boxes, 'labels': labels, 'image_id': image_id}


class TestFrames(Dataset):
    def __init__(self):
        self.paths = sorted((ROOT / 'test' / 'images').rglob('*.jpg'))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]  # .../clip_001/0000001.jpg -> clip_001_0000001
        return TF.to_tensor(Image.open(p).convert('RGB')), {'image_id': f'{p.parent.name}_{p.stem}'}


def collate(batch):
    return tuple(zip(*batch))


train_ds = CropFrames(CocoFrames('train'))
val_ds = CocoFrames('val')
print(NUM_CLASSES, 'classes:', names)

## Model and training

In [ ]:
def build_model(**params):
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=MIN_SIZE, max_size=MAX_SIZE,
        box_detections_per_img=MAX_DET, box_score_thresh=SCORE_THRESH,
        **params,
    )
    # Smaller anchors. Still 3 per location, so the pretrained RPN weights fit.
    model.rpn.anchor_generator = AnchorGenerator(((16,), (32,), (64,), (128,), (256,)), ((0.5, 1.0, 2.0),) * 5)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)  # +1 for background
    return model.to(DEVICE)


def load_model(path, **params):
    """Builds the model and loads weights from a checkpoint, ready for prediction."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    model = build_model(**params)
    model.load_state_dict(state['model'])
    print(f'Loaded {Path(path).name}: {state["epoch"]} epochs')
    return model.eval()


def train_model(name, epochs, dataset, init_ckpt=None, **params):
    """Trains and saves OUT / f'{name}_{epochs}ep.pt' after every epoch, with the optimizer state.
    An existing checkpoint is continued (or used as is if finished); otherwise training starts from
    init_ckpt weights if given, else from COCO weights. Delete the file in OUT to retrain from scratch."""
    torch.manual_seed(0)
    model = build_model(**params)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=LR, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=epochs * len(loader),
                                                    pct_start=0.1, cycle_momentum=False)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')

    path = OUT / f'{name}_{epochs}ep.pt'
    start = 0
    if path.exists():
        state = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state['model'])
        start = state['epoch']
        if start < epochs:
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
        print(f'{name}: loaded checkpoint, {start}/{epochs} epochs done')
        del state
    elif init_ckpt is not None:
        model.load_state_dict(torch.load(init_ckpt, map_location=DEVICE, weights_only=False)['model'])
        print(f'{name}: starting from {Path(init_ckpt).name}')

    for epoch in range(start, epochs):
        model.train()
        total = 0.0
        for images, targets in tqdm(loader, desc=f'{name} epoch {epoch + 1}/{epochs}'):
            images = [im.to(DEVICE) for im in images]
            targets = [{k: t[k].to(DEVICE) for k in ('boxes', 'labels')} for t in targets]
            with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
                loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item()
        print(f'{name} epoch {epoch + 1}: mean loss {total / len(loader):.4f}')

        state = {'model': model.state_dict(), 'epoch': epoch + 1, 'optimizer': optimizer.state_dict(),
                 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict()}
        temp = path.with_suffix('.tmp')
        torch.save(state, temp)
        temp.replace(path)  # a disconnect during saving can't corrupt the checkpoint
    return model.eval()

## Tiled prediction and evaluation

In [ ]:
def tile_windows(W, H, size=TILE, overlap=TILE_OVERLAP):
    """Overlapping (x0, y0, x1, y1) windows covering the frame; the last row and column are aligned to the frame edge."""
    def starts(length):
        if length <= size:
            return [0]
        step = max(1, int(size * (1 - overlap)))
        return list(range(0, length - size, step)) + [length - size]
    if W <= size and H <= size:
        return []  # the full pass already covers it
    return [(x, y, min(x + size, W), min(y + size, H)) for y in starts(H) for x in starts(W)]


def drop_cut_boxes(boxes, window, W, H, margin=EDGE_MARGIN):
    """Mask of tile boxes that don't touch an inner tile edge. Edges on the frame border are fine."""
    x0, y0, x1, y1 = window
    keep = torch.ones(len(boxes), dtype=torch.bool, device=boxes.device)
    if x0 > 0: keep &= boxes[:, 0] > x0 + margin
    if y0 > 0: keep &= boxes[:, 1] > y0 + margin
    if x1 < W: keep &= boxes[:, 2] < x1 - margin
    if y1 < H: keep &= boxes[:, 3] < y1 - margin
    return keep


def to_result(boxes, scores, labels):
    return ((labels - 1).cpu().numpy(), scores.cpu().numpy(), boxes.cpu().numpy())  # 0-based classes


@torch.no_grad()
def predict_tiled(model, dataset, use_tiles=True):
    """Returns (merged, full_only), each {image_id: (cls, scores, boxes)} with xyxy boxes in frame pixels.
    full_only is the plain full-frame prediction, kept to measure what the tiles add."""
    model.eval()
    loader = DataLoader(dataset, batch_size=1, num_workers=2, collate_fn=collate)
    merged, full_only = {}, {}
    for (image,), (target,) in tqdm(loader, desc='predict (tiled)' if use_tiles else 'predict'):
        image = image.to(DEVICE)
        H, W = image.shape[-2:]
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            out = model([image])[0]
        boxes, scores, labels = [out['boxes'].float()], [out['scores'].float()], [out['labels']]
        full_only[target['image_id']] = to_result(boxes[0], scores[0], labels[0])

        windows = tile_windows(W, H) if use_tiles else []
        for k in range(0, len(windows), TILE_BATCH):
            chunk = windows[k:k + TILE_BATCH]
            with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
                outs = model([image[:, y0:y1, x0:x1] for x0, y0, x1, y1 in chunk])
            for (x0, y0, x1, y1), o in zip(chunk, outs):
                b = o['boxes'].float() + torch.tensor([x0, y0, x0, y0], device=DEVICE, dtype=torch.float32)
                keep = drop_cut_boxes(b, (x0, y0, x1, y1), W, H)
                boxes.append(b[keep]); scores.append(o['scores'].float()[keep]); labels.append(o['labels'][keep])

        boxes, scores, labels = torch.cat(boxes), torch.cat(scores), torch.cat(labels)
        keep = batched_nms(boxes, scores, labels, MERGE_IOU)[:MAX_DET]  # sorted by score
        merged[target['image_id']] = to_result(boxes[keep], scores[keep], labels[keep])
    return merged, full_only


def coco_eval(results, dataset):
    cls2cat = {k: cat for cat, k in dataset.cat2cls.items()}
    detections = [
        {'image_id': image_id, 'category_id': cls2cat[int(c)], 'score': float(s),
         'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]}
        for image_id, (cls, scores, boxes) in results.items()
        for c, s, (x1, y1, x2, y2) in zip(cls, scores, boxes)
    ]
    assert detections, 'The model produced no detections'
    ev = COCOeval(dataset.coco, dataset.coco.loadRes(detections), 'bbox')
    ev.params.imgIds = [i for i in dataset.ids if i in results]
    ev.params.maxDets = [1, 10, MAX_DET]
    ev.evaluate()
    ev.accumulate()

    precision = ev.eval['precision']  # [IoU threshold, recall, class, area range, max detections]
    def mean_ap(p):
        return float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    metrics = {'mAP': mean_ap(precision[:, :, :, 0, -1]),
               'AP50': mean_ap(precision[0, :, :, 0, -1]),
               'AP_small': mean_ap(precision[:, :, :, 1, -1]),
               'AP_medium': mean_ap(precision[:, :, :, 2, -1]),
               'AP_large': mean_ap(precision[:, :, :, 3, -1])}
    per_class = {names[dataset.cat2cls[cat]]: mean_ap(precision[:, :, k, 0, -1]) for k, cat in enumerate(ev.params.catIds)}
    return metrics, per_class


def ms_per_image(model, dataset, use_tiles, n=50):
    """Milliseconds per frame for the whole prediction (including tiling and merging) on the first n frames."""
    subset = torch.utils.data.Subset(dataset, range(min(n, len(dataset))))
    predict_tiled(model, torch.utils.data.Subset(dataset, [0]), use_tiles)  # warm-up
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    predict_tiled(model, subset, use_tiles)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / len(subset)

## Train


In [ ]:
model = train_model(RUN_NAME, EPOCHS, train_ds, init_ckpt=INIT_CKPT, **RUN_PARAMS)

In [ ]:
# model = load_model(OUT / f'{RUN_NAME}_{EPOCHS}ep.pt', **RUN_PARAMS)

## Validation

Both rows come from the same model: the full frame alone, and the full frame plus tiles.

In [ ]:
val_tiled, val_full = predict_tiled(model, val_ds)

rows, per_class = {}, {}
for label, results, use_tiles in [('full frame', val_full, False), ('full + tiles', val_tiled, True)]:
    metrics, per_class[label] = coco_eval(results, val_ds)
    ms = ms_per_image(model, val_ds, use_tiles)
    rows[label] = {**metrics, 'ms_per_image': ms, 'FPS': 1000 / ms}

report = pd.DataFrame(rows).T.round(4)
report.to_csv(OUT / f'{RUN_NAME}_metrics.csv')
print('Tiles per frame:', len(tile_windows(*Image.open(val_ds.path(val_ds.ids[0])).size)))
display(report)
display(pd.DataFrame(per_class).sort_values('full + tiles').round(4))

## Submission

In [ ]:
def make_submission(model):
    """Predicts every test image with tiles and returns {image_id: (cls, scores, boxes)}. Nothing is written to disk."""
    merged, _ = predict_tiled(model, TestFrames())  # always every test image, never a subset
    return merged


def save_submission(results, path):
    """Writes predictions as: image_id,PredictionString with "cls score x y w h" for each box.
    Every test image gets a row, and a frame with no boxes gets an empty PredictionString."""
    def to_text(cls, scores, boxes):
        return ' '.join(f'{c} {s:.4f} {x1:.1f} {y1:.1f} {x2 - x1:.1f} {y2 - y1:.1f}'
                        for c, s, (x1, y1, x2, y2) in zip(cls, scores, boxes))

    submission = pd.DataFrame({'image_id': sorted(results)})
    submission['PredictionString'] = submission['image_id'].map(lambda image_id: to_text(*results[image_id]))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    print(f'Saved {path} with {len(submission):,} rows')
    return submission

In [ ]:
prediction = make_submission(model)
submission = save_submission(prediction, OUT / f'submission_{RUN_NAME}.csv')
submission.head()